[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_online_changepoint_detection.ipynb)

# Bayesian Online Changepoint Detection in Python

Many real-world time series exhibit abrupt regime shifts: a financial crisis sends volatility soaring, a sensor begins to drift, or user behaviour changes overnight. Bayesian Online Changepoint Detection (BOCD), introduced by Adams & MacKay (2007), provides a principled, recursive framework for detecting these shifts in real time by maintaining a probability distribution over the current *run length* -- the number of observations since the last changepoint.

Unlike binary detection methods, BOCD outputs a continuous posterior probability that you can threshold to match your risk tolerance. In this notebook we implement BOCD from scratch, apply it to S&P 500 daily log returns, track the posterior probability $P(r_t < 10)$ as a practical detection signal, and compare with the offline PELT algorithm.

In [ ]:
!pip install -q yfinance ruptures

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import ruptures as rpt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from scipy.special import gammaln
import warnings
warnings.filterwarnings("ignore")

## Download S&P 500 Data

In [ ]:
sp = yf.download("^GSPC", start="1984-12-25",
                 auto_adjust=True, progress=False)
sp = sp.loc["1985-01-02":]
close = sp["Close"].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()
signal = log_ret.values.reshape(-1)
dates = log_ret.index
print(f"{len(signal)} trading days, {dates[0].date()} to {dates[-1].date()}")

## BOCD Implementation

We use a Normal-Inverse-Gamma conjugate prior, which gives a Student-t posterior predictive. The `StudentT` class tracks the sufficient statistics for every active run-length hypothesis, and `bocd()` implements the recursive message-passing algorithm from the paper.

In [ ]:
class StudentT:
    """Posterior predictive for Normal-Inverse-Gamma conjugate model."""

    def __init__(self, mu0=0, kappa0=1, alpha0=0.01, beta0=0.01):
        self.mu0 = mu0
        self.kappa0 = kappa0
        self.alpha0 = alpha0
        self.beta0 = beta0
        self.mu = np.array([mu0])
        self.kappa = np.array([kappa0])
        self.alpha = np.array([alpha0])
        self.beta = np.array([beta0])

    def log_pred_prob(self, x):
        """Log predictive probability under each run-length hypothesis."""
        df = 2 * self.alpha
        scale = self.beta * (self.kappa + 1) / (self.alpha * self.kappa)
        return (
            gammaln((df + 1) / 2) - gammaln(df / 2)
            - 0.5 * np.log(np.pi * df * scale)
            - ((df + 1) / 2) * np.log(1 + (x - self.mu)**2 / (df * scale))
        )

    def update(self, x):
        """Bayesian update of sufficient statistics."""
        new_mu = (self.kappa * self.mu + x) / (self.kappa + 1)
        new_kappa = self.kappa + 1
        new_alpha = self.alpha + 0.5
        new_beta = (self.beta
                    + self.kappa * (x - self.mu)**2 / (2 * (self.kappa + 1)))
        self.mu = np.concatenate([[self.mu0], new_mu])
        self.kappa = np.concatenate([[self.kappa0], new_kappa])
        self.alpha = np.concatenate([[self.alpha0], new_alpha])
        self.beta = np.concatenate([[self.beta0], new_beta])

In [ ]:
def bocd(data, hazard_rate=250, short_threshold=10):
    """Bayesian Online Changepoint Detection (Adams & MacKay 2007).

    Returns R (full run-length matrix), maxes (MAP run length),
    and p_short (probability that the current regime is younger
    than short_threshold days).
    """
    T = len(data)
    R = np.zeros((T + 1, T + 1))
    R[0, 0] = 1.0
    model = StudentT()
    h = 1.0 / hazard_rate
    maxes = np.zeros(T)
    p_short = np.zeros(T)

    for t in range(T):
        x = data[t]
        pred = np.exp(model.log_pred_prob(x))
        R[1:t+2, t+1] = R[:t+1, t] * pred * (1 - h)
        R[0, t+1] = np.sum(R[:t+1, t] * pred * h)
        evidence = R[:t+2, t+1].sum()
        if evidence > 0:
            R[:t+2, t+1] /= evidence
        model.update(x)
        maxes[t] = np.argmax(R[:t+2, t+1])
        p_short[t] = R[:min(short_threshold, t+2), t+1].sum()

    return R, maxes, p_short

## Run BOCD

In [ ]:
R, maxes, p_short = bocd(signal, hazard_rate=250)

# Detect changepoints: where MAP run length drops sharply
changepoints = []
for t in range(1, len(signal)):
    if maxes[t] < 5 and maxes[t-1] > 20:
        changepoints.append(t)
print(f"Found {len(changepoints)} changepoints")

## Run-Length Posterior Heatmap

In [ ]:
max_rl = 500
T = len(signal)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2]})

# Top panel: S&P 500 price
ax0 = axes[0]
ax0.plot(dates, close.loc[dates].values, color="black", linewidth=0.6)
for cp in changepoints:
    ax0.axvline(dates[cp], color="red", alpha=0.5, linewidth=0.7)
ax0.set_ylabel("S&P 500 Close")
ax0.set_title("Bayesian Online Changepoint Detection on S&P 500 Daily Returns")

# Bottom panel: run-length posterior heatmap
ax1 = axes[1]
# Subsample for plotting performance
step = max(1, T // 2000)
t_idx = np.arange(0, T, step)
R_sub = R[:max_rl, 1:T+1][:, t_idx]
R_sub = np.clip(R_sub, 1e-12, None)

date_nums = mdates.date2num(dates)
extent = [date_nums[t_idx[0]], date_nums[t_idx[-1]], 0, max_rl]
im = ax1.imshow(R_sub, aspect="auto", origin="lower", cmap="magma",
                norm=LogNorm(vmin=1e-6, vmax=R_sub.max()), extent=extent,
                interpolation="nearest")
ax1.xaxis_date()
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator(5))
ax1.set_ylabel("Run Length")
ax1.set_xlabel("Date")
fig.colorbar(im, ax=ax1, label="P(run length | data)", shrink=0.8)

plt.tight_layout()
plt.show()

## Posterior Probability: P(run length < 10)

Rather than relying on the MAP run length alone, we can summarise the posterior as $P(r_t < 10)$: the probability that the current regime started fewer than 10 trading days ago. This gives a continuous signal that rises as evidence of a regime shift accumulates. You choose a threshold (5%, 25%, 50%) based on your tolerance for false positives versus detection speed.

In [ ]:
# Full timeline: S&P 500 on top, P(rl < 10) on bottom
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1]})

ax0 = axes[0]
ax0.plot(dates, close.loc[dates].values, color="black", linewidth=0.6)
for cp in changepoints:
    ax0.axvline(dates[cp], color="red", alpha=0.35, linewidth=0.7)
ax0.set_ylabel("S&P 500 Close")
ax0.set_title("S&P 500 with BOCD Changepoints")

ax1 = axes[1]
ax1.plot(dates, p_short * 100, color="steelblue", linewidth=0.6)
ax1.axhline(5, color="orange", linestyle="--", alpha=0.7, label="5% threshold")
ax1.axhline(50, color="red", linestyle="--", alpha=0.7, label="50% threshold")
ax1.set_ylabel("P(run length < 10) [%]")
ax1.set_xlabel("Date")
ax1.set_ylim(-2, 105)
ax1.legend(loc="upper left")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator(5))

plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on three events: COVID, Liberation Day, Iran War
zoom_events = [
    ("COVID-19", "2020-02-10", "2020-04-01"),
    ("Liberation Day Tariffs", "2025-03-15", "2025-05-01"),
    ("Iran War", "2026-02-15", "2026-04-01"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, (title, start, end) in zip(axes, zoom_events):
    mask = (dates >= start) & (dates <= end)
    d = dates[mask]
    ps = p_short[np.array(mask)] * 100

    ax.plot(d, ps, color="steelblue", linewidth=1.5)
    ax.axhline(5, color="orange", linestyle="--", alpha=0.7, label="5%")
    ax.axhline(25, color="purple", linestyle="--", alpha=0.7, label="25%")
    ax.axhline(50, color="red", linestyle="--", alpha=0.7, label="50%")
    ax.set_title(title, fontsize=11)
    ax.set_ylim(-2, 105)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.tick_params(axis="x", rotation=30)

axes[0].set_ylabel("P(run length < 10) [%]")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Posterior probability around three events", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Threshold-based detection timing for key events
key_events = {
    "Black Monday": "1987-10-14",
    "GFC (Bear Stearns)": "2008-03-14",
    "US Downgrade": "2011-08-04",
    "China Devaluation": "2015-08-20",
    "Volmageddon": "2018-02-02",
    "COVID": "2020-02-20",
    "Liberation Day": "2025-04-02",
    "Iran War": "2026-02-28",
}
thresholds = [0.05, 0.25, 0.50]

rows = []
for event_name, start_str in key_events.items():
    start_date = pd.Timestamp(start_str)
    mask = dates >= start_date
    if not mask.any():
        continue
    start_idx = np.where(mask)[0][0]
    row = {"Event": event_name, "Sell-off Start": start_str}
    for thr in thresholds:
        detected = False
        for t in range(start_idx, min(start_idx + 30, len(p_short))):
            if p_short[t] >= thr:
                lag = (dates[t] - start_date).days
                row[f"{int(thr*100)}% Threshold"] = f"{dates[t].date()} (+{lag}d)"
                detected = True
                break
        if not detected:
            row[f"{int(thr*100)}% Threshold"] = "—"
    # MAP flip
    detected = False
    for t in range(max(1, start_idx), min(start_idx + 30, len(maxes))):
        if maxes[t] < 5 and maxes[t-1] > 20:
            lag = (dates[t] - start_date).days
            row["MAP Flip"] = f"{dates[t].date()} (+{lag}d)"
            detected = True
            break
    if not detected:
        row["MAP Flip"] = "—"
    rows.append(row)

timing_df = pd.DataFrame(rows)
timing_df

## Changepoints Aligned with Financial Events

In [ ]:
events = {
    "1987-10-19": "Black Monday",
    "2000-03-10": "Dot-com",
    "2001-09-17": "9/11",
    "2008-09-15": "Lehman",
    "2010-05-06": "Flash Crash",
    "2011-08-05": "US Downgrade",
    "2015-08-24": "China Deval.",
    "2018-02-05": "Volmageddon",
    "2020-03-09": "COVID",
    "2022-06-13": "Rate Hikes",
    "2025-04-02": "Liberation Day",
    "2026-02-28": "Iran War",
}

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(dates, close.loc[dates].values, color="black", linewidth=0.6,
        label="S&P 500")

for cp in changepoints:
    ax.axvline(dates[cp], color="red", alpha=0.35, linewidth=0.7)

# Annotate known events
y_positions = np.linspace(0.55, 0.95, len(events))
for i, (date_str, label) in enumerate(events.items()):
    event_date = pd.Timestamp(date_str)
    if event_date < dates[0] or event_date > dates[-1]:
        continue
    ax.axvline(event_date, color="blue", linestyle="--", alpha=0.6,
               linewidth=0.9)
    ax.annotate(label, xy=(event_date, ax.get_ylim()[1]),
                xytext=(event_date, ax.get_ylim()[1] * y_positions[i]),
                fontsize=8, color="blue", ha="center",
                arrowprops=dict(arrowstyle="-", color="blue", alpha=0.3))

ax.set_title("BOCD Changepoints (red) vs Known Financial Events (blue dashed)")
ax.set_ylabel("S&P 500 Close")
ax.set_xlabel("Date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(5))
plt.tight_layout()
plt.show()

## Exercises

1. **Different hazard function.** The implementation above uses a constant hazard rate $h = 1/\lambda$. Modify `bocd()` to accept a time-varying hazard function, e.g. a linearly increasing hazard $h(t) = t / (\lambda^2)$, and observe how the detected changepoints shift.

2. **Different dataset.** Apply BOCD to Bitcoin daily returns (ticker `BTC-USD` on Yahoo Finance). How does the algorithm behave on a much more volatile asset? Do you need to adjust the hazard rate or prior hyperparameters?

3. **Multivariate extension.** The tutorial feeds BOCD a single feature (daily log return). You could construct a 2D input of (daily log return, 21-day rolling volatility) to detect shifts in both level and stability simultaneously. A 3D or 4D input could combine short-window and long-window volatilities (e.g. 5-day and 63-day). The required change is replacing the Normal-Inverse-Gamma prior with a [Normal-Inverse-Wishart](https://en.wikipedia.org/wiki/Normal-inverse-Wishart_distribution) prior, which jointly models a mean vector and covariance matrix. The predictive distribution becomes a multivariate Student-t. The message-passing loop stays identical; only the sufficient statistics change.

In [ ]:
hazard_rates = [100, 250, 500, 1000]

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

for ax, hr in zip(axes, hazard_rates):
    _, mx, _ = bocd(signal, hazard_rate=hr)
    cps = []
    for t in range(1, len(signal)):
        if mx[t] < 5 and mx[t-1] > 20:
            cps.append(t)

    ax.plot(dates, close.loc[dates].values, color="black", linewidth=0.6)
    for cp in cps:
        ax.axvline(dates[cp], color="red", alpha=0.5, linewidth=0.7)
    ax.set_ylabel("Close")
    ax.set_title(f"Hazard rate = 1/{hr}  ({len(cps)} changepoints)",
                 fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))

axes[-1].set_xlabel("Date")
fig.suptitle("Hazard Rate Sensitivity", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## BOCD vs PELT Comparison

In [ ]:
# Run PELT (offline)
algo = rpt.Pelt(model="normal", min_size=10, jump=5).fit(signal)
pelt_bkps = algo.predict(pen=20)
pelt_cps = [b for b in pelt_bkps if b < len(signal)]  # exclude endpoint

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# BOCD
axes[0].plot(dates, close.loc[dates].values, color="black", linewidth=0.6)
for cp in changepoints:
    axes[0].axvline(dates[cp], color="red", alpha=0.5, linewidth=0.7)
axes[0].set_title(f"BOCD (online) -- {len(changepoints)} changepoints",
                   fontsize=11)
axes[0].set_ylabel("S&P 500 Close")

# PELT
axes[1].plot(dates, close.loc[dates].values, color="black", linewidth=0.6)
for cp in pelt_cps:
    axes[1].axvline(dates[cp], color="green", alpha=0.5, linewidth=0.7)
axes[1].set_title(f"PELT (offline, pen=20) -- {len(pelt_cps)} changepoints",
                   fontsize=11)
axes[1].set_ylabel("S&P 500 Close")
axes[1].set_xlabel("Date")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))

plt.tight_layout()
plt.show()

## Exercises

1. **Different hazard function.** The implementation above uses a constant hazard rate $h = 1/\lambda$. Modify `bocd()` to accept a time-varying hazard function, e.g. a linearly increasing hazard $h(t) = t / (\lambda^2)$, and observe how the detected changepoints shift.

2. **Different dataset.** Apply BOCD to Bitcoin daily returns (ticker `BTC-USD` on Yahoo Finance). How does the algorithm behave on a much more volatile asset? Do you need to adjust the hazard rate or prior hyperparameters?

3. **Multivariate extension.** Implement a multivariate version of BOCD using the Normal-Inverse-Wishart conjugate prior. Apply it to a bivariate signal (e.g. S&P 500 and VIX returns jointly) and compare the detected changepoints with the univariate results.